# Actividad 1 

Nuria Arroyo  

In [43]:
import numpy as np
import pandas as pd


#todos los helpers juntos para no repetir el mismo codigo en cada ejercicio
def tabla_de_valores(valores_por_tiempo, decimales=3):
    """convierte un diccionario tipo valores[t][estado] en una tabla bonita."""
    tabla = pd.DataFrame(valores_por_tiempo).T
    tabla.index.name = "t"
    return tabla.round(decimales)


def tabla_de_politica(politica_por_tiempo):
    """convierte policy[t][estado] en tabla para leerla rapido."""
    tabla = pd.DataFrame(politica_por_tiempo).T
    tabla.index.name = "t"
    return tabla


def tabla_transiciones(estados, transiciones):
    """arma una tabla del kernel q(y|x,a) sin escribirlo a mano."""
    filas = []

    for (estado_actual, accion), probs in transiciones.items():
        fila = {
            "estado_actual": estado_actual,
            "accion": accion,
        }

        for estado_siguiente in estados:
            fila[f"q_{estado_siguiente}"] = probs.get(estado_siguiente, 0.0)

        filas.append(fila)

    return pd.DataFrame(filas)


def tabla_valores_inmediatos(valores_inmediatos, nombre_columna):
    """pasa recompensas o costos esperados a una tabla explicita."""
    filas = []

    for (estado_actual, accion), valor in valores_inmediatos.items():
        filas.append({
            "estado_actual": estado_actual,
            "accion": accion,
            nombre_columna: valor,
        })

    return pd.DataFrame(filas).round(3)


def comparar_politicas(estados, valores_por_politica, decimales=3):
    """compara policies usando el valor inicial v_0 de cada estado."""
    datos = {"estado_inicial": estados}

    for nombre_politica, valores in valores_por_politica.items():
        datos[nombre_politica] = [valores[0][estado] for estado in estados]

    return pd.DataFrame(datos).round(decimales)


def evaluar_politica_horizonte(estados, transiciones, valores_inmediatos, politica, horizonte):
    """
    evalua una policy estacionaria en horizonte finito.

    sirve igual para recompensas y costos. la diferencia es como se interpreta el valor.
    """
    valores = {horizonte: {estado: 0.0 for estado in estados}}

    for periodo in reversed(range(horizonte)):
        valores[periodo] = {}

        for estado_actual in estados:
            valor_estado = 0.0

            for accion, prob_accion in politica[estado_actual].items():
                valor_futuro = sum(
                    prob_transicion * valores[periodo + 1][estado_siguiente]
                    for estado_siguiente, prob_transicion in transiciones[(estado_actual, accion)].items()
                )

                valor_accion = valores_inmediatos[(estado_actual, accion)] + valor_futuro
                valor_estado += prob_accion * valor_accion

            valores[periodo][estado_actual] = valor_estado

    return valores


def resolver_backward_induction(estados, acciones_por_estado, transiciones,
                                valores_inmediatos, horizonte, objetivo):
    """
    resuelve bellman por backward induction.

    objetivo='max' para recompensas y objetivo='min' para costos.
    """
    if objetivo not in {"max", "min"}:
        raise ValueError("objetivo debe ser 'max' o 'min'")

    valores = {horizonte: {estado: 0.0 for estado in estados}}
    politica_optima = {}

    for periodo in reversed(range(horizonte)):
        valores[periodo] = {}
        politica_optima[periodo] = {}

        for estado_actual in estados:
            valores_accion = {}

            for accion in acciones_por_estado[estado_actual]:
                valor_futuro = sum(
                    prob_transicion * valores[periodo + 1][estado_siguiente]
                    for estado_siguiente, prob_transicion in transiciones[(estado_actual, accion)].items()
                )

                valores_accion[accion] = valores_inmediatos[(estado_actual, accion)] + valor_futuro

            if objetivo == "max":
                mejor_accion = max(valores_accion, key=valores_accion.get)
            else:
                mejor_accion = min(valores_accion, key=valores_accion.get)

            politica_optima[periodo][estado_actual] = mejor_accion
            valores[periodo][estado_actual] = valores_accion[mejor_accion]

    return valores, politica_optima


def detalle_bellman(estados, acciones_por_estado, transiciones, valores_inmediatos,
                    valores, horizonte, nombre_inmediato, nombre_total):
    """muestra el valor que bellman le dio a cada accion antes de escoger."""
    filas = []

    for periodo in reversed(range(horizonte)):
        for estado_actual in estados:
            for accion in acciones_por_estado[estado_actual]:
                valor_futuro = sum(
                    prob_transicion * valores[periodo + 1][estado_siguiente]
                    for estado_siguiente, prob_transicion in transiciones[(estado_actual, accion)].items()
                )
                valor_total = valores_inmediatos[(estado_actual, accion)] + valor_futuro

                filas.append({
                    "t": periodo,
                    "estado_actual": estado_actual,
                    "accion": accion,
                    nombre_inmediato: valores_inmediatos[(estado_actual, accion)],
                    "valor_futuro_esperado": valor_futuro,
                    nombre_total: valor_total,
                })

    return pd.DataFrame(filas).round(3)


def simular_politica_estacionaria(transiciones, valores_inmediatos, politica,
                                  estado_inicial, horizonte, n_sim=5000, seed=123):
    """simula una policy estacionaria cuando el payoff depende de (estado, accion)."""
    rng = np.random.default_rng(seed)
    totales = []
    trayectorias = []

    for _ in range(n_sim):
        estado_actual = estado_inicial
        total_trayectoria = 0.0
        trayectoria = []

        for periodo in range(horizonte):
            acciones_posibles = list(politica[estado_actual].keys())
            probs_accion = list(politica[estado_actual].values())
            accion = rng.choice(acciones_posibles, p=probs_accion)

            siguientes_estados = list(transiciones[(estado_actual, accion)].keys())
            probs_transicion = list(transiciones[(estado_actual, accion)].values())
            estado_siguiente = rng.choice(siguientes_estados, p=probs_transicion)

            valor_inmediato = valores_inmediatos[(estado_actual, accion)]
            total_trayectoria += valor_inmediato

            trayectoria.append({
                "t": periodo,
                "estado_actual": estado_actual,
                "accion": accion,
                "valor_inmediato": valor_inmediato,
                "estado_siguiente": estado_siguiente,
            })

            estado_actual = estado_siguiente

        totales.append(total_trayectoria)
        trayectorias.append(trayectoria)

    return float(np.mean(totales)), totales, trayectorias


def simular_politica_tiempo(transiciones, valores_inmediatos, politica_por_tiempo,
                            estado_inicial, horizonte, n_sim=5000, seed=123):
    """simula una policy que cambia por periodo, como la que sale de bellman."""
    rng = np.random.default_rng(seed)
    totales = []
    trayectorias = []

    for _ in range(n_sim):
        estado_actual = estado_inicial
        total_trayectoria = 0.0
        trayectoria = []

        for periodo in range(horizonte):
            accion = politica_por_tiempo[periodo][estado_actual]
            siguientes_estados = list(transiciones[(estado_actual, accion)].keys())
            probs_transicion = list(transiciones[(estado_actual, accion)].values())
            estado_siguiente = rng.choice(siguientes_estados, p=probs_transicion)

            valor_inmediato = valores_inmediatos[(estado_actual, accion)]
            total_trayectoria += valor_inmediato

            trayectoria.append({
                "t": periodo,
                "estado_actual": estado_actual,
                "accion": accion,
                "valor_inmediato": valor_inmediato,
                "estado_siguiente": estado_siguiente,
            })

            estado_actual = estado_siguiente

        totales.append(total_trayectoria)
        trayectorias.append(trayectoria)

    return float(np.mean(totales)), totales, trayectorias


# helpers de inventario, porque los ejercicios 2 y 4 comparten casi todo
def acciones_factibles_inventario(estados, orden_maxima, capacidad):
    """acciones validas: no pedir mas de la orden maxima ni pasar la capacidad."""
    return {
        inventario: [
            orden for orden in range(orden_maxima + 1)
            if inventario + orden <= capacidad
        ]
        for inventario in estados
    }


def siguiente_inventario(inventario_actual, orden, demanda):
    """modelo de ventas perdidas: si falta inventario, no se va a negativo."""
    return max(0, inventario_actual + orden - demanda)


def costo_periodo_inventario(inventario_actual, orden, demanda,
                             costo_fijo, costo_unitario,
                             costo_mantener, penalizacion_faltante):
    """costo real de un periodo cuando ya vimos la demanda."""
    inventario_pre_demanda = inventario_actual + orden
    inventario_final = siguiente_inventario(inventario_actual, orden, demanda)
    unidades_faltantes = max(0, demanda - inventario_pre_demanda)

    costo_de_ordenar = costo_fijo if orden > 0 else 0
    costo_por_unidades = costo_unitario * orden
    costo_por_guardar = costo_mantener * inventario_final
    costo_por_faltante = penalizacion_faltante * unidades_faltantes

    return costo_de_ordenar + costo_por_unidades + costo_por_guardar + costo_por_faltante


def construir_mdp_inventario(estados, prob_demanda, orden_maxima, capacidad,
                             costo_fijo, costo_unitario,
                             costo_mantener, penalizacion_faltante):
    """construye acciones, kernel y costo esperado desde la distribucion de demanda."""
    acciones_por_estado = acciones_factibles_inventario(estados, orden_maxima, capacidad)
    transiciones = {}
    costos_esperados = {}

    for inventario_actual in estados:
        for orden in acciones_por_estado[inventario_actual]:
            probs_siguiente = {estado: 0.0 for estado in estados}
            costo_esperado = 0.0

            for demanda, prob_demanda_valor in prob_demanda.items():
                inventario_siguiente = siguiente_inventario(inventario_actual, orden, demanda)
                probs_siguiente[inventario_siguiente] += prob_demanda_valor

                costo = costo_periodo_inventario(
                    inventario_actual=inventario_actual,
                    orden=orden,
                    demanda=demanda,
                    costo_fijo=costo_fijo,
                    costo_unitario=costo_unitario,
                    costo_mantener=costo_mantener,
                    penalizacion_faltante=penalizacion_faltante,
                )
                costo_esperado += prob_demanda_valor * costo

            transiciones[(inventario_actual, orden)] = probs_siguiente
            costos_esperados[(inventario_actual, orden)] = costo_esperado

    return acciones_por_estado, transiciones, costos_esperados


def elegir_accion_desde_regla(regla_accion, rng):
    """la regla puede ser una accion fija o un dict con probabilidades."""
    if isinstance(regla_accion, dict):
        acciones = list(regla_accion.keys())
        probs_accion = list(regla_accion.values())
        return int(rng.choice(acciones, p=probs_accion))

    return int(regla_accion)


def simular_politica_inventario(politica, estado_inicial, horizonte, prob_demanda,
                                costo_fijo, costo_unitario, costo_mantener,
                                penalizacion_faltante, n_sim=5000, seed=123,
                                politica_depende_tiempo=False):
    """simula inventario con demanda real, para validar contra el valor esperado."""
    rng = np.random.default_rng(seed)
    demandas = list(prob_demanda.keys())
    probs_demanda = list(prob_demanda.values())
    costos_totales = []
    trayectorias = []

    for _ in range(n_sim):
        inventario_actual = estado_inicial
        costo_total = 0.0
        trayectoria = []

        for periodo in range(horizonte):
            if politica_depende_tiempo:
                regla_accion = politica[periodo][inventario_actual]
            else:
                regla_accion = politica[inventario_actual]

            orden = elegir_accion_desde_regla(regla_accion, rng)
            demanda = int(rng.choice(demandas, p=probs_demanda))
            inventario_siguiente = siguiente_inventario(inventario_actual, orden, demanda)

            costo = costo_periodo_inventario(
                inventario_actual=inventario_actual,
                orden=orden,
                demanda=demanda,
                costo_fijo=costo_fijo,
                costo_unitario=costo_unitario,
                costo_mantener=costo_mantener,
                penalizacion_faltante=penalizacion_faltante,
            )

            trayectoria.append({
                "t": periodo,
                "inventario_inicial": inventario_actual,
                "orden": orden,
                "demanda": demanda,
                "inventario_final": inventario_siguiente,
                "costo": costo,
            })

            costo_total += costo
            inventario_actual = inventario_siguiente

        costos_totales.append(costo_total)
        trayectorias.append(trayectoria)

    return float(np.mean(costos_totales)), costos_totales, trayectorias


# Ejercicio 1: MDP de horizonte finito

##  Modelado formal

El problema se modela como un Proceso de Decisión de Markov de horizonte finito. En este caso, el sistema puede estar en tres estados:

 $$
X=\{s_1,s_2,s_3\}
 $$

donde:

 $$
s_1=\text{sistema en estado favorable}
 $$

 $$
s_2=\text{sistema en estado intermedio}
 $$

 $$
s_3=\text{sistema en estado crítico}
 $$

La idea es que, en cada periodo, observamos el estado actual del sistema y tomamos una acción. Esa acción genera una recompensa inmediata y también afecta las probabilidades de transición hacia el siguiente estado.



## Acciones admisibles

Las acciones disponibles dependen del estado actual:

 $$
A(s_1)=\{a_1,a_2\}
 $$

 $$
A(s_2)=\{a_1,a_2\}
 $$

 $$
A(s_3)=\{a_2\}
 $$

Esto significa que en los estados favorable e intermedio sí podemos elegir entre dos acciones, mientras que en el estado crítico solamente está disponible la acción  $a_2 $. Esto tiene sentido porque, cuando el sistema ya está en estado crítico, la capacidad de decisión es más limitada.


## Recompensas inmediatas

La función de recompensa está dada por:

 $$
r(s_1,a_1)=6
 $$

 $$
r(s_1,a_2)=4
 $$

 $$
r(s_2,a_1)=3
 $$

 $$
r(s_2,a_2)=5
 $$

 $$
r(s_3,a_2)=-2
 $$

Estas recompensas representan el beneficio inmediato de tomar cierta acción en cierto estado. Por ejemplo, en  $ s_1 $, la acción  $a_1 $ da mayor recompensa inmediata que  $a_2 $. En  $s_2 $, ocurre lo contrario:  $a_2 $ da mayor recompensa inmediata que  $a_1 $. En  $s_3 $, la única acción disponible genera una recompensa negativa, lo cual refleja que estar en estado crítico representa una pérdida o un costo para el sistema.



---

## 4. Kernel de transición

El kernel de transición describe la probabilidad de pasar de un estado actual  $s $ a un estado futuro  $s' $, dado que se toma una acción  $a $.

Formalmente:

 $$
Q(s'|s,a)=P(X_{t+1}=s' \mid X_t=s,A_t=a)
 $$

Las probabilidades de transición son:

 $$
P(\cdot|s_1,a_1)=(0.7,0.3,0)
 $$

 $$
P(\cdot|s_1,a_2)=(0.4,0.6,0)
 $$

 $$
P(\cdot|s_2,a_1)=(0.2,0.5,0.3)
 $$

 $$
P(\cdot|s_2,a_2)=(0.1,0.6,0.3)
 $$

 $$
P(\cdot|s_3,a_2)=(0,0.4,0.6)
 $$

Cada vector está ordenado respecto a:

 $$
(s_1,s_2,s_3)
 $$

Por ejemplo:

 $$
P(\cdot|s_1,a_1)=(0.7,0.3,0)
 $$

significa que, si el sistema está en  $s_1 $ y se toma  $a_1 $, entonces:

 $$
P(s_1|s_1,a_1)=0.7
 $$

 $$
P(s_2|s_1,a_1)=0.3
 $$

 $$
P(s_3|s_1,a_1)=0
 $$




##  Horizonte finito

El horizonte del problema es:

 $$
N=4
 $$

Esto significa que se evalúan cuatro periodos de decisión. Como es un problema de horizonte finito, se define una condición terminal:

 $$
V_4(s)=0
 $$

para todo:

 $$
s\in X
 $$

Es decir, después del último periodo ya no se acumula recompensa adicional.



## Ecuación de Bellman para horizonte finito

La ecuación de Bellman para un problema de maximización con horizonte finito es:

 $$
V_t(s)=
\max_{a\in A(s)}
\left\{
r(s,a)+
\sum_{s'\in X}P(s'|s,a)V_{t+1}(s')
\right\}
 $$

para:

 $$
t=N-1,N-2,\dots,0
 $$

con condición terminal:

 $$
V_N(s)=0
 $$

Esta ecuación dice que el valor de estar en un estado no depende solamente de la recompensa inmediata, sino también del valor esperado de los estados futuros. Por eso la decisión no debe verse de manera aislada, sino como parte de una secuencia de decisiones.



## Evaluación de una política fija

En este ejercicio se pide evaluar políticas propuestas. Cuando la política ya está dada, no se toma el máximo de Bellman. En su lugar, se calcula el valor esperado siguiendo esa política.

Para una política Markoviana  $\pi $, la evaluación se hace con:

 $$
V_t^\pi(s)=
\sum_{a\in A(s)}
\pi_t(a|s)
\left[
r(s,a)+
\sum_{s'\in X}P(s'|s,a)V_{t+1}^\pi(s')
\right]
 $$

donde:

 $$
\pi_t(a|s)=P(A_t=a|X_t=s)
 $$

Si la política es determinista, una acción tiene probabilidad 1 y las demás probabilidad 0. Si la política es aleatorizada, varias acciones pueden tener probabilidades positivas.





# Política determinista propuesta

Propongo la siguiente política determinista:

$$
\pi_D(s_1)=a_1
$$

$$
\pi_D(s_2)=a_2
$$

$$
\pi_D(s_3)=a_2
$$

La lógica de esta política es elegir la acción con mejor recompensa inmediata en los estados donde hay más de una opción. En $s_1$, $a_1$ da recompensa 6, mientras que $a_2$ da recompensa 4. En $s_2$, $a_2$ da recompensa 5, mientras que $a_1$ da recompensa 3. En $s_3$, no hay decisión real porque solamente existe $a_2$.

---

## Evaluación de la política determinista

La condición terminal es:

$$
V_4^{\pi_D}(s)=0
$$

para todo estado $s$.

La ecuación de evaluación se reduce a:

$$
V_t^{\pi_D}(s)=
r(s,\pi_D(s))+
\sum_{s'\in X}
P(s'|s,\pi_D(s))V_{t+1}^{\pi_D}(s')
$$

porque la acción está completamente determinada por el estado.

---

### Cálculo en $t=3$

Como:

$$
V_4^{\pi_D}(s_1)=V_4^{\pi_D}(s_2)=V_4^{\pi_D}(s_3)=0
$$

entonces:

$$
V_3^{\pi_D}(s_1)
=
6+0.7(0)+0.3(0)+0(0)=6
$$

$$
V_3^{\pi_D}(s_2)
=
5+0.1(0)+0.6(0)+0.3(0)=5
$$

$$
V_3^{\pi_D}(s_3)
=
-2+0(0)+0.4(0)+0.6(0)=-2
$$

---

### Cálculo en $t=2$

$$
V_2^{\pi_D}(s_1)
=
6+0.7V_3(s_1)+0.3V_3(s_2)+0V_3(s_3)
$$

$$
V_2^{\pi_D}(s_1)
=
6+0.7(6)+0.3(5)+0(-2)
$$

$$
V_2^{\pi_D}(s_1)=11.7
$$

$$
V_2^{\pi_D}(s_2)
=
5+0.1V_3(s_1)+0.6V_3(s_2)+0.3V_3(s_3)
$$

$$
V_2^{\pi_D}(s_2)
=
5+0.1(6)+0.6(5)+0.3(-2)
$$

$$
V_2^{\pi_D}(s_2)=8
$$

$$
V_2^{\pi_D}(s_3)
=
-2+0V_3(s_1)+0.4V_3(s_2)+0.6V_3(s_3)
$$

$$
V_2^{\pi_D}(s_3)
=
-2+0(6)+0.4(5)+0.6(-2)
$$

$$
V_2^{\pi_D}(s_3)=-1.2
$$

---

### Cálculo en $t=1$

$$
V_1^{\pi_D}(s_1)
=
6+0.7(11.7)+0.3(8)+0(-1.2)
$$

$$
V_1^{\pi_D}(s_1)=16.59
$$

$$
V_1^{\pi_D}(s_2)
=
5+0.1(11.7)+0.6(8)+0.3(-1.2)
$$

$$
V_1^{\pi_D}(s_2)=10.61
$$

$$
V_1^{\pi_D}(s_3)
=
-2+0(11.7)+0.4(8)+0.6(-1.2)
$$

$$
V_1^{\pi_D}(s_3)=0.48
$$

---

### Cálculo en $t=0$

$$
V_0^{\pi_D}(s_1)
=
6+0.7(16.59)+0.3(10.61)+0(0.48)
$$

$$
V_0^{\pi_D}(s_1)=20.796
$$

$$
V_0^{\pi_D}(s_2)
=
5+0.1(16.59)+0.6(10.61)+0.3(0.48)
$$

$$
V_0^{\pi_D}(s_2)=13.169
$$

$$
V_0^{\pi_D}(s_3)
=
-2+0(16.59)+0.4(10.61)+0.6(0.48)
$$

$$
V_0^{\pi_D}(s_3)=2.532
$$

---

## Tabla final de valores para la política determinista

| Tiempo | $V_t(s_1)$ | $V_t(s_2)$ | $V_t(s_3)$ |
|---|---:|---:|---:|
| $t=4$ | 0.000 | 0.000 | 0.000 |
| $t=3$ | 6.000 | 5.000 | -2.000 |
| $t=2$ | 11.700 | 8.000 | -1.200 |
| $t=1$ | 16.590 | 10.610 | 0.480 |
| $t=0$ | 20.796 | 13.169 | 2.532 |

Si el sistema inicia en $s_1$, la recompensa total esperada bajo esta política es:

$$
V_0^{\pi_D}(s_1)=20.796
$$

---


# Política Markoviana aleatorizada propuesta

Propongo la siguiente política aleatorizada:

$$
\pi_R(a_1|s_1)=0.7
$$

$$
\pi_R(a_2|s_1)=0.3
$$

$$
\pi_R(a_1|s_2)=0.4
$$

$$
\pi_R(a_2|s_2)=0.6
$$

$$
\pi_R(a_2|s_3)=1
$$

Esta política sigue siendo Markoviana porque la decisión depende únicamente del estado actual. La diferencia es que, en lugar de elegir siempre una misma acción, se permite cierta aleatorización.

---

## Evaluación de la política aleatorizada

La ecuación de evaluación es:

$$
V_t^{\pi_R}(s)=
\sum_{a\in A(s)}
\pi_R(a|s)
\left[
r(s,a)+
\sum_{s'\in X}P(s'|s,a)V_{t+1}^{\pi_R}(s')
\right]
$$

con condición terminal:

$$
V_4^{\pi_R}(s)=0
$$

---

## Tabla final de valores para la política aleatorizada

| Tiempo | $V_t(s_1)$ | $V_t(s_2)$ | $V_t(s_3)$ |
|---|---:|---:|---:|
| $t=4$ | 0.000 | 0.000 | 0.000 |
| $t=3$ | 5.400 | 4.200 | -2.000 |
| $t=2$ | 10.332 | 6.708 | -1.520 |
| $t=1$ | 14.319 | 8.947 | -0.229 |
| $t=0$ | 17.624 | 11.146 | 1.442 |

Si el sistema inicia en $s_1$, la recompensa total esperada bajo esta política es:

$$
V_0^{\pi_R}(s_1)=17.624
$$

---

# Comparación de políticas

| Estado inicial | Política determinista | Política aleatorizada |
|---|---:|---:|
| $s_1$ | 20.796 | 17.624 |
| $s_2$ | 13.169 | 11.146 |
| $s_3$ | 2.532 | 1.442 |

La política determinista propuesta tiene mayor recompensa total esperada en los tres estados iniciales. Esto ocurre porque la política determinista elige de forma consistente las acciones que generan mejor desempeño esperado en este caso. La política aleatorizada introduce variabilidad y, aunque sigue siendo válida, a veces toma acciones menos favorables.

---

# Verificación con Monte Carlo

Para verificar los resultados se puede usar simulación Monte Carlo. La idea es generar muchas trayectorias siguiendo una política fija, sumar las recompensas obtenidas en cada trayectoria y calcular el promedio.

El retorno simulado de una trayectoria es:

$$
G=\sum_{t=0}^{N-1} r(X_t,A_t)
$$

Después de simular $M$ trayectorias, el estimador Monte Carlo es:

$$
\hat{V}^{\pi}(s)=
\frac{1}{M}
\sum_{i=1}^{M}
G_i
$$

donde $G_i$ es la recompensa total obtenida en la trayectoria $i$.

Con $M=5000$ trayectorias, el promedio simulado debe acercarse a los valores calculados analíticamente. No tiene que coincidir exactamente porque Monte Carlo depende de simulación aleatoria, pero sí debe estar cerca.

---

# Interpretación final

Este ejercicio muestra cómo modelar un problema de decisión secuencial usando un MDP. El estado representa la condición del sistema, la acción representa la decisión tomada, la recompensa mide el beneficio inmediato y el kernel de transición describe la evolución probabilística del sistema.

La parte importante es que una decisión no se evalúa solamente por su recompensa inmediata. También se debe considerar a qué estados puede llevar en el futuro. Por eso se usa la ecuación de Bellman y se calcula el valor esperado hacia atrás.

En este caso, la política determinista propuesta tiene mejor recompensa total esperada que la aleatorizada. Esto sugiere que, para este modelo específico, tomar decisiones consistentes según el estado produce mejor desempeño que introducir aleatorización.

In [2]:
# datos base del ejercicio 1
estados_ej1 = ["s1", "s2", "s3"]

acciones_ej1 = {
    "s1": ["a1", "a2"],
    "s2": ["a1", "a2"],
    "s3": ["a2"],
}

recompensas_ej1 = {
    ("s1", "a1"): 6,
    ("s1", "a2"): 4,
    ("s2", "a1"): 3,
    ("s2", "a2"): 5,
    ("s3", "a2"): -2,
}

transiciones_ej1 = {
    ("s1", "a1"): {"s1": 0.7, "s2": 0.3, "s3": 0.0},
    ("s1", "a2"): {"s1": 0.4, "s2": 0.6, "s3": 0.0},
    ("s2", "a1"): {"s1": 0.2, "s2": 0.5, "s3": 0.3},
    ("s2", "a2"): {"s1": 0.1, "s2": 0.6, "s3": 0.3},
    ("s3", "a2"): {"s1": 0.0, "s2": 0.4, "s3": 0.6},
}

# policy determinista: en cada estado solo una accion tiene probabilidad 1
politica_det_ej1 = {
    "s1": {"a1": 1.0},
    "s2": {"a2": 1.0},
    "s3": {"a2": 1.0},
}

# policy random: misma idea markoviana, pero con mezcla de acciones
politica_rand_ej1 = {
    "s1": {"a1": 0.7, "a2": 0.3},
    "s2": {"a1": 0.4, "a2": 0.6},
    "s3": {"a2": 1.0},
}

horizonte_ej1 = 4
estado_inicial_ej1 = "s1"

valores_det_ej1 = evaluar_politica_horizonte(
    estados=estados_ej1,
    transiciones=transiciones_ej1,
    valores_inmediatos=recompensas_ej1,
    politica=politica_det_ej1,
    horizonte=horizonte_ej1,
)

valores_rand_ej1 = evaluar_politica_horizonte(
    estados=estados_ej1,
    transiciones=transiciones_ej1,
    valores_inmediatos=recompensas_ej1,
    politica=politica_rand_ej1,
    horizonte=horizonte_ej1,
)

print("policy determinista")
display(tabla_de_valores(valores_det_ej1))

print("policy aleatorizada")
display(tabla_de_valores(valores_rand_ej1))

print("comparacion por estado inicial")
display(comparar_politicas(
    estados_ej1,
    {
        "determinista": valores_det_ej1,
        "aleatorizada": valores_rand_ej1,
    },
))

media_mc_det_ej1, retornos_det_ej1, trayectorias_det_ej1 = simular_politica_estacionaria(
    transiciones=transiciones_ej1,
    valores_inmediatos=recompensas_ej1,
    politica=politica_det_ej1,
    estado_inicial=estado_inicial_ej1,
    horizonte=horizonte_ej1,
    n_sim=5000,
    seed=123,
)

media_mc_rand_ej1, retornos_rand_ej1, trayectorias_rand_ej1 = simular_politica_estacionaria(
    transiciones=transiciones_ej1,
    valores_inmediatos=recompensas_ej1,
    politica=politica_rand_ej1,
    estado_inicial=estado_inicial_ej1,
    horizonte=horizonte_ej1,
    n_sim=5000,
    seed=123,
)

resumen_mc_ej1 = pd.DataFrame({
    "policy": ["determinista", "aleatorizada"],
    "valor_analitico_desde_s1": [
        valores_det_ej1[0][estado_inicial_ej1],
        valores_rand_ej1[0][estado_inicial_ej1],
    ],
    "valor_monte_carlo_desde_s1": [media_mc_det_ej1, media_mc_rand_ej1],
})

resumen_mc_ej1.round(3)


policy determinista


,s1,s2,s3
t,,,
4,0.000,0.000,0.000
3,6.000,5.000,-2.000
2,11.700,8.000,-1.200
1,16.590,10.610,0.480
0,20.796,13.169,2.532


policy aleatorizada


,s1,s2,s3
t,,,
4,0.000,0.000,0.000
3,5.400,4.200,-2.000
2,10.332,6.708,-1.520
1,14.319,8.947,-0.229
0,17.624,11.146,1.442


comparacion por estado inicial


,estado_inicial,determinista,aleatorizada
0,s1,20.796,17.624
1,s2,13.169,11.146
2,s3,2.532,1.442


,policy,valor_analitico_desde_s1,valor_monte_carlo_desde_s1
0,determinista,20.796,20.824
1,aleatorizada,17.624,17.652


# Ejercicio 2: Sistema de inventario como MDP

##  Modelado formal del MDP

El problema se puede modelar como un Proceso de Decisión de Markov de horizonte finito. En este caso, el estado representa el inventario disponible al inicio de cada periodo.

El espacio de estados es:

$$
X=\{0,1,2,3,4\}
$$

donde $X_t=x$ representa el número de unidades disponibles al inicio del periodo $t$.





## Acciones admisibles

La acción $a_t$ representa cuántas unidades se ordenan en el periodo $t$. Las acciones posibles son:

$$
a_t\in\{0,1,2\}
$$

pero están sujetas a la restricción de capacidad:

$$
X_t+a_t\leq 4
$$

Por lo tanto, el conjunto de acciones admisibles depende del estado:

$$
A(x)=\{a\in\{0,1,2\}:x+a\leq 4\}
$$

Esto significa que no siempre puedo ordenar 2 unidades. Si ya tengo mucho inventario, la acción se restringe para no superar la capacidad máxima de 4 unidades.




Lista de acciones explcitas para cada estado:

s0: $A(0)=\{0,1,2\}$

s1: $A(1)=\{0,1,2\}$

s2: $A(2)=\{0,1,2\}$

s3: $A(3)=\{0,1\}$

s4: $A(4)=\{0\}$


##  Demanda aleatoria

La demanda aleatoria puede tomar tres valores:

$$
D_t\in\{0,1,2\}
$$

con probabilidades:

$$
P(D_t=0)=0.2
$$

$$
P(D_t=1)=0.5
$$

$$
P(D_t=2)=0.3
$$

La demanda es la parte aleatoria del problema. En el ejercicio 1 ya teníamos directamente las probabilidades de transición, pero aquí las transiciones se construyen a partir de la demanda.



##  Dinámica del inventario

Después de observar el inventario $X_t$ y ordenar $a_t$, el inventario disponible antes de la demanda es:

$$
Y_t=X_t+a_t
$$

Después ocurre la demanda $D_t$. Como el modelo es de ventas perdidas, si la demanda supera el inventario disponible, no se acumula inventario negativo. Por eso:

$$
X_{t+1}=\max\{0,X_t+a_t-D_t\}
$$

o equivalentemente:

$$
X_{t+1}=(X_t+a_t-D_t)^+
$$

donde:

$$
(z)^+=\max\{0,z\}
$$

---



##  Ley de transición

El kernel de transición se obtiene a partir de la demanda:

$$
Q(y|x,a)=P(X_{t+1}=y\mid X_t=x,A_t=a)
$$

Como:

$$
X_{t+1}=\max\{0,x+a-D_t\}
$$

entonces:

$$
Q(y|x,a)=
\sum_{d\in\{0,1,2\}}
P(D_t=d)\mathbf{1}_{\{y=\max(0,x+a-d)\}}
$$

Esta expresión dice que la probabilidad de llegar a un estado $y$ se obtiene sumando las probabilidades de todas las demandas que generan ese mismo inventario final.



##  Kernel de transición explícito para cada estado y acción:

| Estado $x$ | Acción $a$ | $Q(0\mid x,a)$ | $Q(1\mid x,a)$ | $Q(2\mid x,a)$ | $Q(3\mid x,a)$ | $Q(4\mid x,a)$ |
|---|---|---:|---:|---:|---:|---:|
| 0 | 0 | 1.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 0 | 1 | 0.8 | 0.2 | 0.0 | 0.0 | 0.0 |
| 0 | 2 | 0.3 | 0.5 | 0.2 | 0.0 | 0.0 |
| 1 | 0 | 0.8 | 0.2 | 0.0 | 0.0 | 0.0 |
| 1 | 1 | 0.3 | 0.5 | 0.2 | 0.0 | 0.0 |
| 1 | 2 | 0.0 | 0.3 | 0.5 | 0.2 | 0.0 |
| 2 | 0 | 0.3 | 0.5 | 0.2 | 0.0 | 0.0 |
| 2 | 1 | 0.0 | 0.3 | 0.5 | 0.2 | 0.0 |
| 2 | 2 | 0.0 | 0.0 | 0.3 | 0.5 | 0.2 |
| 3 | 0 | 0.0 | 0.3 | 0.5 | 0.2 | 0.0 |
| 3 | 1 | 0.0 | 0.0 | 0.3 | 0.5 | 0.2 |
| 4 | 0 | 0.0 | 0.0 | 0.3 | 0.5 | 0.2 |
---



##  Costos del problema

El problema no tiene recompensas, sino costos. El costo de cada periodo tiene cuatro componentes.

Primero, el costo fijo de ordenar:

$$
K\mathbf{1}_{\{a>0\}}
$$

donde:

$$
K=2
$$

Segundo, el costo unitario de ordenar:

$$
ca
$$

donde:

$$
c=1
$$

Tercero, el costo de mantener inventario sobrante:

$$
hX_{t+1}
$$

donde:

$$
h=1
$$

Cuarto, la penalización por demanda no satisfecha:

$$
p(D_t-(x+a))^+
$$

donde:

$$
p=4
$$

Por lo tanto, el costo por periodo es:

$$
C(x,a,D)=K\mathbf{1}_{\{a>0\}}+ca+h\max\{0,x+a-D\}+p(D-(x+a))^+
$$

---



##  Costo esperado inmediato

Como la demanda es aleatoria, el costo inmediato esperado es:

$$
\bar{c}(x,a)=
E[C(x,a,D)]
$$

Entonces:

$$
\bar{c}(x,a)=
\sum_{d\in\{0,1,2\}}
P(D=d)
\left[
K\mathbf{1}_{\{a>0\}}
+ca
+h\max\{0,x+a-d\}
+p(d-(x+a))^+
\right]
$$

Este costo esperado es el que se usa dentro de la ecuación de Bellman.




## Ecuación de Bellman para minimizar costo

Como el objetivo es minimizar costo total esperado, la ecuación de Bellman es:

$$
V_t(x)=
\min_{a\in A(x)}
\left\{
\bar{c}(x,a)
+
\sum_{y\in X}
Q(y|x,a)V_{t+1}(y)
\right\}
$$

con condición terminal:

$$
V_N(x)=0
$$

En este ejercicio:

$$
N=3
$$

por lo que:

$$
V_3(x)=0
$$

---

## Evaluación de una política determinista

Propongo una política determinista de tipo umbral. La idea es mantener al menos 2 unidades disponibles, porque la demanda máxima posible es 2.

La política es:

$$
\pi_D(0)=2
$$

$$
\pi_D(1)=1
$$

$$
\pi_D(2)=0
$$

$$
\pi_D(3)=0
$$

$$
\pi_D(4)=0
$$

Es decir:

| Estado $x$ | Acción $\pi_D(x)$ |
|---|---:|
| 0 | 2 |
| 1 | 1 |
| 2 | 0 |
| 3 | 0 |
| 4 | 0 |

La interpretación es que si el inventario está muy bajo, ordeno para protegerme contra demanda alta. Si ya tengo 2 o más unidades, no ordeno para evitar costos innecesarios de pedido y mantenimiento.

---

##  Evaluación de una política aleatorizada

También se puede proponer una política Markoviana aleatorizada. Por ejemplo:

$$
\pi_R(2|0)=0.8,\qquad \pi_R(1|0)=0.2
$$

$$
\pi_R(1|1)=0.7,\qquad \pi_R(0|1)=0.3
$$

$$
\pi_R(0|2)=0.8,\qquad \pi_R(1|2)=0.2
$$

$$
\pi_R(0|3)=0.9,\qquad \pi_R(1|3)=0.1
$$

$$
\pi_R(0|4)=1
$$

Esta política sigue siendo Markoviana porque la distribución de acciones depende solamente del estado actual. La diferencia es que no siempre toma la misma acción, sino que introduce cierta aleatorización.


##  Evaluación de política fija

Para una política fija $\pi$, ya no minimizamos sobre las acciones. En su lugar, promediamos sobre las acciones que la política puede tomar:

$$
V_t^\pi(x)=
\sum_{a\in A(x)}
\pi(a|x)
\left[
\bar{c}(x,a)
+
\sum_{y\in X}
Q(y|x,a)V_{t+1}^\pi(y)
\right]
$$

con:

$$
V_N^\pi(x)=0
$$

Este valor representa el costo total esperado desde el estado $x$, siguiendo la política $\pi$.

---

##  Simulación Monte Carlo

Para validar los resultados, se simulan muchas trayectorias del sistema de inventario.

En cada trayectoria:

1. Se parte de un estado inicial.
2. Se elige una acción según la política.
3. Se simula una demanda $D_t$.
4. Se calcula el costo real del periodo.
5. Se actualiza el inventario con:

$$
X_{t+1}=\max\{0,X_t+a_t-D_t\}
$$

6. Se repite hasta completar el horizonte $N=3$.
7. Se promedian los costos totales simulados.

El costo total de una trayectoria es:

$$
G=\sum_{t=0}^{N-1}C(X_t,A_t,D_t)
$$

El estimador Monte Carlo es:

$$
\hat{V}^{\pi}(x)=
\frac{1}{M}
\sum_{i=1}^{M}G_i
$$

donde $M$ es el número de trayectorias simuladas.

---

##  Interpretación

Este problema representa una situación de inventario donde se debe decidir cuánto ordenar antes de observar la demanda. Ordenar demasiado puede generar costos de pedido y mantenimiento, pero ordenar poco puede generar faltantes.

La política determinista propuesta busca protegerse contra faltantes manteniendo al menos 2 unidades disponibles. Sin embargo, esto no necesariamente implica que sea óptima, porque a veces ordenar puede ser más caro que aceptar cierto riesgo de faltante.

La política aleatorizada permite comparar qué pasa cuando las decisiones no son completamente fijas. Esto puede representar incertidumbre operativa, exploración o variabilidad en la toma de decisiones.

El objetivo final es comparar ambas políticas en términos de costo total esperado: la mejor política será la que tenga menor costo esperado.

In [3]:
# datos base del ejercicio 2
estados_ej2 = [0, 1, 2, 3, 4]
horizonte_ej2 = 3
capacidad_ej2 = 4
orden_maxima_ej2 = 2

prob_demanda_ej2 = {
    0: 0.2,
    1: 0.5,
    2: 0.3,
}

costo_fijo_ej2 = 2
costo_unitario_ej2 = 1
costo_mantener_ej2 = 1
penalizacion_faltante_ej2 = 4

acciones_ej2, transiciones_ej2, costos_esperados_ej2 = construir_mdp_inventario(
    estados=estados_ej2,
    prob_demanda=prob_demanda_ej2,
    orden_maxima=orden_maxima_ej2,
    capacidad=capacidad_ej2,
    costo_fijo=costo_fijo_ej2,
    costo_unitario=costo_unitario_ej2,
    costo_mantener=costo_mantener_ej2,
    penalizacion_faltante=penalizacion_faltante_ej2,
)

# aqui se ve clarito que acciones se pueden usar sin romper la capacidad
acciones_df_ej2 = pd.DataFrame({
    "estado_actual": list(acciones_ej2.keys()),
    "acciones_factibles": list(acciones_ej2.values()),
})

display(acciones_df_ej2)

tabla_kernel_ej2 = tabla_transiciones(estados_ej2, transiciones_ej2)
tabla_kernel_ej2


,estado_actual,acciones_factibles
0,0,"[0, 1, 2]"
1,1,"[0, 1, 2]"
2,2,"[0, 1, 2]"
3,3,"[0, 1]"
4,4,[0]


,estado_actual,accion,q_0,q_1,q_2,q_3,q_4
0,0,0,1.0,0.0,0.0,0.0,0.0
1,0,1,0.8,0.2,0.0,0.0,0.0
2,0,2,0.3,0.5,0.2,0.0,0.0
3,1,0,0.8,0.2,0.0,0.0,0.0
4,1,1,0.3,0.5,0.2,0.0,0.0
5,1,2,0.0,0.3,0.5,0.2,0.0
6,2,0,0.3,0.5,0.2,0.0,0.0
7,2,1,0.0,0.3,0.5,0.2,0.0
8,2,2,0.0,0.0,0.3,0.5,0.2
9,3,0,0.0,0.3,0.5,0.2,0.0


In [4]:
# costos esperados inmediatos, ya promediados sobre la demanda
costos_esperados_df_ej2 = tabla_valores_inmediatos(
    valores_inmediatos=costos_esperados_ej2,
    nombre_columna="costo_esperado",
)

costos_esperados_df_ej2


,estado_actual,accion,costo_esperado
0,0,0,4.4
1,0,1,4.4
2,0,2,4.9
3,1,0,1.4
4,1,1,3.9
5,1,2,5.9
6,2,0,0.9
7,2,1,4.9
8,2,2,6.9
9,3,0,1.9


In [5]:
# policy determinista tipo umbral: si hay poco inventario, pido para llegar a 2
politica_det_ej2 = {
    0: {2: 1.0},
    1: {1: 1.0},
    2: {0: 1.0},
    3: {0: 1.0},
    4: {0: 1.0},
}

# policy random: mete un poco de variacion, pero sin usar acciones no factibles
politica_rand_ej2 = {
    0: {2: 0.8, 1: 0.2},
    1: {1: 0.7, 0: 0.3},
    2: {0: 0.8, 1: 0.2},
    3: {0: 0.9, 1: 0.1},
    4: {0: 1.0},
}

valores_det_ej2 = evaluar_politica_horizonte(
    estados=estados_ej2,
    transiciones=transiciones_ej2,
    valores_inmediatos=costos_esperados_ej2,
    politica=politica_det_ej2,
    horizonte=horizonte_ej2,
)

valores_rand_ej2 = evaluar_politica_horizonte(
    estados=estados_ej2,
    transiciones=transiciones_ej2,
    valores_inmediatos=costos_esperados_ej2,
    politica=politica_rand_ej2,
    horizonte=horizonte_ej2,
)

print("costo total esperado con policy determinista")
display(tabla_de_valores(valores_det_ej2))

print("costo total esperado con policy aleatorizada")
display(tabla_de_valores(valores_rand_ej2))

print("comparacion de costos iniciales")
display(comparar_politicas(
    estados_ej2,
    {
        "determinista": valores_det_ej2,
        "aleatorizada": valores_rand_ej2,
    },
))


costo total esperado con policy determinista


,0,1,2,3,4
t,,,,,
3,0.0,0.0,0.0,0.00,0.00
2,4.9,3.9,0.9,1.90,2.90
1,8.5,7.5,4.5,3.90,4.70
0,12.1,11.1,8.1,7.18,7.14


costo total esperado con policy aleatorizada


,0,1,2,3,4
t,,,,,
3,0.000,0.000,0.000,0.000,0.000
2,4.800,3.150,1.700,2.300,2.900
1,8.378,6.840,4.835,4.554,5.140
0,11.934,10.401,8.296,7.618,7.655


comparacion de costos iniciales


,estado_inicial,determinista,aleatorizada
0,0,12.10,11.934
1,1,11.10,10.401
2,2,8.10,8.296
3,3,7.18,7.618
4,4,7.14,7.655


In [6]:
# monte carlo usa demanda simulada, no solo el costo esperado ya promediado
estado_inicial_ej2 = 0
n_sim_ej2 = 5000

media_mc_det_ej2, costos_mc_det_ej2, trayectorias_det_ej2 = simular_politica_inventario(
    politica=politica_det_ej2,
    estado_inicial=estado_inicial_ej2,
    horizonte=horizonte_ej2,
    prob_demanda=prob_demanda_ej2,
    costo_fijo=costo_fijo_ej2,
    costo_unitario=costo_unitario_ej2,
    costo_mantener=costo_mantener_ej2,
    penalizacion_faltante=penalizacion_faltante_ej2,
    n_sim=n_sim_ej2,
    seed=123,
)

media_mc_rand_ej2, costos_mc_rand_ej2, trayectorias_rand_ej2 = simular_politica_inventario(
    politica=politica_rand_ej2,
    estado_inicial=estado_inicial_ej2,
    horizonte=horizonte_ej2,
    prob_demanda=prob_demanda_ej2,
    costo_fijo=costo_fijo_ej2,
    costo_unitario=costo_unitario_ej2,
    costo_mantener=costo_mantener_ej2,
    penalizacion_faltante=penalizacion_faltante_ej2,
    n_sim=n_sim_ej2,
    seed=123,
)

resumen_mc_ej2 = pd.DataFrame({
    "policy": ["determinista", "aleatorizada"],
    "costo_analitico_desde_0": [
        valores_det_ej2[0][estado_inicial_ej2],
        valores_rand_ej2[0][estado_inicial_ej2],
    ],
    "costo_monte_carlo_desde_0": [media_mc_det_ej2, media_mc_rand_ej2],
})

resumen_mc_ej2.round(3)


,policy,costo_analitico_desde_0,costo_monte_carlo_desde_0
0,determinista,12.100,12.130
1,aleatorizada,11.934,11.918


# Ejercicio 3: MDP de horizonte finito con intervención

## 1. Aclaración importante: estado no es recompensa

En este ejercicio puede haber confusión porque los estados están escritos como números:

$$
X=\{0,1,2\}
$$

Entonces, cuando aparece $x$, $x$ representa el estado actual del sistema.

Pero la recompensa es otra variable distinta. La recompensa se calcula a partir del estado y de la acción, pero no es lo mismo que el estado.

En un MDP tenemos:

$$
X_t=\text{estado en el tiempo }t
$$

$$
A_t=\text{acción tomada en el tiempo }t
$$

$$
R_t=\text{recompensa recibida en el tiempo }t
$$

Por lo tanto:

$$
X_t \neq R_t
$$

aunque la recompensa pueda depender de $X_t$.

---



## 2. Espacio de estados

El espacio de estados es:

$$
X=\{0,1,2\}
$$

Esto significa que el sistema solamente puede estar en alguno de esos tres estados.

Por ejemplo, el sistema puede estar en estado 0, estado 1 o estado 2. No puede estar en estado $-2$.

---

## 3. Acciones

En cada estado se puede elegir una de dos acciones:

$$
A(x)=\{0,1\}
$$

para todo:

$$
x\in X
$$

La acción:

$$
a=0
$$

representa no intervenir.

La acción:

$$
a=1
$$

representa intervenir.

---



## 4. Recompensas inmediatas

La recompensa inmediata está definida como:

$$
r(x,0)=x
$$

$$
r(x,1)=x-2
$$

La primera expresión significa que, si no se interviene, la recompensa inmediata es igual al valor numérico del estado.

La segunda expresión significa que, si se interviene, se paga un costo de 2 unidades.



---

##  Tabla de recompensas

Evaluando las recompensas para cada estado y acción:

| Estado $x$ | Acción $a$ | Recompensa $r(x,a)$ |
|---|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | -2 |
| 1 | 0 | 1 |
| 1 | 1 | -1 |
| 2 | 0 | 2 |
| 2 | 1 | 0 |

Es importante notar que las recompensas pueden ser negativas. Por ejemplo:

$$
r(0,1)=0-2=-2
$$

Esto no significa que el estado sea $-2$. Significa que, estando en el estado 0, si se toma la acción de intervenir, se recibe una recompensa de $-2$, que puede interpretarse como un costo.

El estado sigue siendo 0, 1 o 2. La recompensa es una cantidad aparte.

---

##  Transiciones

Las probabilidades de transición describen a qué estado pasa el sistema después de tomar una acción.

Para la acción $a=0$, es decir, no intervenir:

$$
P(\cdot|0,0)=(0.6,0.4,0)
$$

$$
P(\cdot|1,0)=(0,0.7,0.3)
$$

$$
P(\cdot|2,0)=(0,0,1)
$$

Para la acción $a=1$, es decir, intervenir:

$$
P(\cdot|0,1)=(1,0,0)
$$

$$
P(\cdot|1,1)=(0.8,0.2,0)
$$

$$
P(\cdot|2,1)=(0.5,0.5,0)
$$

Cada vector está ordenado respecto a:

$$
(0,1,2)
$$

---



##  MDP formal

El MDP se define como:

$$
(X,A,Q,r)
$$

donde:

$$
X=\{0,1,2\}
$$

$$
A(x)=\{0,1\}
$$

$$
Q(y|x,a)=P(X_{t+1}=y|X_t=x,A_t=a)
$$

y:

$$
r(x,0)=x
$$

$$
r(x,1)=x-2
$$

El horizonte finito es:

$$
N=5
$$

---



## Ecuación de Bellman

El objetivo es maximizar la recompensa total esperada. Por lo tanto, la ecuación de Bellman es:

$$
V_t(x)=
\max_{a\in A(x)}
\left\{
r(x,a)+
\sum_{y\in X}P(y|x,a)V_{t+1}(y)
\right\}
$$

con condición terminal:

$$
V_N(x)=0
$$

Como:

$$
N=5
$$

entonces:

$$
V_5(x)=0
$$

para todo:

$$
x\in X
$$



## Backward induction

La inducción hacia atrás consiste en resolver desde el final del horizonte hacia el inicio.

Primero sabemos que:

$$
V_5(x)=0
$$




Como se requeire en el ejercicio, N-1 es 4, entonces se empieza a resolver desde $t=4$ hacia atrás.


En la última etapa:

$$
V_5(x)=0
$$

Entonces Bellman se vuelve:

$$
V_4(x)=
\max_{a\in\{0,1\}}
\{r(x,a)\}
$$

porque ya no hay futuro que considerar.

Para $x=0$:

$$
r(0,0)=0
$$

$$
r(0,1)=-2
$$

Entonces:

$$
V_4(0)=0
$$

y:

$$
\pi_4^*(0)=0
$$

Para $x=1$:

$$
r(1,0)=1
$$

$$
r(1,1)=-1
$$

Entonces:

$$
V_4(1)=1
$$

y:

$$
\pi_4^*(1)=0
$$

Para $x=2$:

$$
r(2,0)=2
$$

$$
r(2,1)=0
$$

Entonces:

$$
V_4(2)=2
$$

y:

$$
\pi_4^*(2)=0
$$

Por lo tanto:

$$
V_4=(0,1,2)
$$

En la última etapa no conviene intervenir porque intervenir tiene un costo inmediato y ya no queda futuro para compensar ese costo.

---

## Algoritmo general

Para cada tiempo:

$$
t=4,3,2,1,0
$$

y para cada estado:

$$
x\in\{0,1,2\}
$$

se comparan las dos acciones.

Para cada acción se calcula:

$$
r(x,a)+
\sum_{y\in X}P(y|x,a)V_{t+1}(y)
$$

Después se elige la acción que da el mayor valor.

La política óptima es:

$$
\pi_t^*(x)=
\arg\max_{a\in A(x)}
\left\{
r(x,a)+
\sum_{y\in X}P(y|x,a)V_{t+1}(y)
\right\}
$$


## Interpretación

La política óptima puede cambiar dependiendo del periodo porque el horizonte es finito. Intervenir tiene un costo inmediato de 2 unidades, por lo que cerca del final casi nunca conviene intervenir.

Sin embargo, al inicio del horizonte, intervenir podría ser conveniente si mejora suficientemente los estados futuros. Es decir, aunque intervenir tenga una recompensa inmediata menor, puede ser útil si reduce la probabilidad de quedar en estados menos convenientes más adelante.

La lógica completa es:

$$
\text{valor de una acción}
=
\text{recompensa inmediata}
+
\text{valor esperado del futuro}
$$

Por eso el problema no se resuelve viendo solamente la recompensa inmediata, sino considerando también las consecuencias futuras de cada acción.

In [7]:
# datos base del ejercicio 3
estados_ej3 = [0, 1, 2]

acciones_ej3 = {
    0: [0, 1],
    1: [0, 1],
    2: [0, 1],
}

horizonte_ej3 = 5

# recompensa: no intervenir da x, intervenir cuesta 2 unidades de reward
recompensas_ej3 = {
    (estado_actual, accion): estado_actual if accion == 0 else estado_actual - 2
    for estado_actual in estados_ej3
    for accion in acciones_ej3[estado_actual]
}

recompensas_df_ej3 = tabla_valores_inmediatos(
    valores_inmediatos=recompensas_ej3,
    nombre_columna="recompensa",
)

recompensas_df_ej3


,estado_actual,accion,recompensa
0,0,0,0
1,0,1,-2
2,1,0,1
3,1,1,-1
4,2,0,2
5,2,1,0


In [8]:
# kernel dado en el enunciado del ejercicio 3
transiciones_ej3 = {
    (0, 0): {0: 0.6, 1: 0.4, 2: 0.0},
    (1, 0): {0: 0.0, 1: 0.7, 2: 0.3},
    (2, 0): {0: 0.0, 1: 0.0, 2: 1.0},
    (0, 1): {0: 1.0, 1: 0.0, 2: 0.0},
    (1, 1): {0: 0.8, 1: 0.2, 2: 0.0},
    (2, 1): {0: 0.5, 1: 0.5, 2: 0.0},
}

tabla_kernel_ej3 = tabla_transiciones(estados_ej3, transiciones_ej3)
tabla_kernel_ej3


,estado_actual,accion,q_0,q_1,q_2
0,0,0,0.6,0.4,0.0
1,1,0,0.0,0.7,0.3
2,2,0,0.0,0.0,1.0
3,0,1,1.0,0.0,0.0
4,1,1,0.8,0.2,0.0
5,2,1,0.5,0.5,0.0


In [9]:
# backward induction para maximizar recompensa total esperada
valores_opt_ej3, politica_opt_ej3 = resolver_backward_induction(
    estados=estados_ej3,
    acciones_por_estado=acciones_ej3,
    transiciones=transiciones_ej3,
    valores_inmediatos=recompensas_ej3,
    horizonte=horizonte_ej3,
    objetivo="max",
)

print("funciones de valor optimas")
display(tabla_de_valores(valores_opt_ej3))

print("policy optima por periodo")
display(tabla_de_politica(politica_opt_ej3))


funciones de valor optimas


,0,1,2
t,,,
5,0.000,0.000,0.0
4,0.000,1.000,2.0
3,0.400,2.300,4.0
2,1.160,3.810,6.0
1,2.220,5.467,8.0
0,3.519,7.227,10.0


policy optima por periodo


,0,1,2
t,,,
4,0,0,0
3,0,0,0
2,0,0,0
1,0,0,0
0,0,0,0


In [10]:
# detalle de todas las comparaciones que hace bellman
bellman_detalle_ej3 = detalle_bellman(
    estados=estados_ej3,
    acciones_por_estado=acciones_ej3,
    transiciones=transiciones_ej3,
    valores_inmediatos=recompensas_ej3,
    valores=valores_opt_ej3,
    horizonte=horizonte_ej3,
    nombre_inmediato="recompensa_inmediata",
    nombre_total="valor_total_accion",
)

bellman_detalle_ej3


,t,estado_actual,accion,recompensa_inmediata,valor_futuro_esperado,valor_total_accion
0,4,0,0,0,0.000,0.000
1,4,0,1,-2,0.000,-2.000
2,4,1,0,1,0.000,1.000
3,4,1,1,-1,0.000,-1.000
4,4,2,0,2,0.000,2.000
5,4,2,1,0,0.000,0.000
6,3,0,0,0,0.400,0.400
7,3,0,1,-2,0.000,-2.000
8,3,1,0,1,1.300,2.300
9,3,1,1,-1,0.200,-0.800


In [11]:
# validacion monte carlo de la policy optima dependiente del tiempo
estado_inicial_ej3 = 0
n_sim_ej3 = 5000

media_mc_opt_ej3, retornos_mc_ej3, trayectorias_opt_ej3 = simular_politica_tiempo(
    transiciones=transiciones_ej3,
    valores_inmediatos=recompensas_ej3,
    politica_por_tiempo=politica_opt_ej3,
    estado_inicial=estado_inicial_ej3,
    horizonte=horizonte_ej3,
    n_sim=n_sim_ej3,
    seed=123,
)

resumen_mc_ej3 = pd.DataFrame({
    "estado_inicial": [estado_inicial_ej3],
    "valor_analitico": [valores_opt_ej3[0][estado_inicial_ej3]],
    "valor_monte_carlo": [media_mc_opt_ej3],
})

resumen_mc_ej3.round(3)


,estado_inicial,valor_analitico,valor_monte_carlo
0,0,3.519,3.468


In [12]:
# una trayectoria example para revisar que la simulacion tenga sentido
pd.DataFrame(trayectorias_opt_ej3[0])


,t,estado_actual,accion,valor_inmediato,estado_siguiente
0,0,0,0,0,1
1,1,1,0,1,1
2,2,1,0,1,1
3,3,1,0,1,1
4,4,1,0,1,1


In [13]:
# resumen del valor optimo desde cada estado inicial
resumen_estados_ej3 = pd.DataFrame({
    "estado_inicial": estados_ej3,
    "valor_optimo_analitico": [valores_opt_ej3[0][estado] for estado in estados_ej3],
})

resumen_estados_ej3.round(3)


,estado_inicial,valor_optimo_analitico
0,0,3.519
1,1,7.227
2,2,10.000


# Ejercicio 4: Sistema de inventario con Backward Induction


#  Modelado formal del MDP

El problema se modela como un MDP de horizonte finito:

$$
(X,A,Q,c)
$$

donde:

- \(X\): espacio de estados,
- \(A\): acciones admisibles,
- \(Q\): kernel de transición,
- \(c\): costo inmediato esperado.

---

#  Espacio de estados

El estado representa el inventario disponible al inicio del periodo:

$$
X_t\in\{0,1,2,3\}
$$

El valor de \(X_t\) indica cuántas unidades hay disponibles antes de ordenar y antes de observar la demanda.

---

#  Acciones

La acción representa cuántas unidades ordenar:

$$
a_t\in\{0,1,2\}
$$

sujeto a:

$$
X_t+a_t\leq 3
$$

Por lo tanto, las acciones admisibles dependen del estado actual:

$$
A(x)=\{a\in\{0,1,2\}:x+a\leq 3\}
$$

Esto es exactamente igual al Ejercicio 2, solamente cambia la capacidad máxima.

---

#  Demanda aleatoria

La demanda puede tomar valores:

$$
D_t\in\{0,1,2\}
$$

con probabilidades:

$$
P(D_t=0)=0.3
$$

$$
P(D_t=1)=0.4
$$

$$
P(D_t=2)=0.3
$$

La demanda es la fuente de aleatoriedad del sistema y es lo que induce las probabilidades de transición.

---

# Dinámica del inventario

La dinámica es exactamente la misma del Ejercicio 2.

Después de ordenar:

$$
Y_t=X_t+a_t
$$

Después ocurre la demanda \(D_t\). Como el modelo es de ventas perdidas:

$$
X_{t+1}=\max\{0,X_t+a_t-D_t\}
$$

o equivalentemente:

$$
X_{t+1}=(X_t+a_t-D_t)^+
$$

donde:

$$
(z)^+=\max\{0,z\}
$$

---

#  Kernel de transición

El kernel de transición se construye igual que en el Ejercicio 2:

$$
Q(y|x,a)=P(X_{t+1}=y|X_t=x,A_t=a)
$$

Como:

$$
X_{t+1}=\max\{0,x+a-D_t\}
$$

entonces:

$$
Q(y|x,a)
=
\sum_{d\in\{0,1,2\}}
P(D=d)\mathbf{1}_{\{y=\max(0,x+a-d)\}}
$$





#  Costos

Los costos por periodo también son iguales al Ejercicio 2, excepto que ahora la penalización es:

$$
p=5
$$

Los componentes son:

### Costo fijo de ordenar

$$
K\mathbf{1}_{\{a>0\}}
$$

con:

$$
K=2
$$

---

### Costo unitario

$$
ca
$$

con:

$$
c=1
$$

---

### Costo de mantenimiento

$$
hX_{t+1}
$$

con:

$$
h=1
$$

---

### Penalización por faltante

$$
p(D-(x+a))^+
$$

con:

$$
p=5
$$


# Costo total por periodo

El costo real de un periodo es:

$$
C(x,a,D)
=
K\mathbf{1}_{\{a>0\}}
+
ca
+
h\max\{0,x+a-D\}
+
p(D-(x+a))^+
$$



#  Costo esperado inmediato

Como la demanda es aleatoria, Bellman usa el costo esperado:

$$
\bar c(x,a)=E[C(x,a,D)]
$$

Entonces:

$$
\bar c(x,a)
=
\sum_{d\in\{0,1,2\}}
P(D=d)
\left[
K\mathbf{1}_{\{a>0\}}
+
ca
+
h\max\{0,x+a-d\}
+
p(d-(x+a))^+
\right]
$$

Esto también es exactamente igual al Ejercicio 2.

Ecuación de Bellman:

Ahora sí buscamos la política óptima. Como queremos minimizar costo esperado total:

$$
V_t(x)=
\min_{a\in A(x)}
\left\{
\bar c(x,a)
+
\sum_{y\in X}Q(y|x,a)V_{t+1}(y)
\right\}
$$

con condición terminal:

$$
V_4(x)=0
$$



#  Backward Induction

La inducción hacia atrás consiste en resolver el problema desde el final hacia el inicio.

Primero sabemos:

$$
V_4(x)=0
$$

porque después del horizonte ya no hay más costos.

Después calculamos:

$$
V_3(x)
$$

luego:

$$
V_2(x)
$$

luego:

$$
V_1(x)
$$

y finalmente:

$$
V_0(x)
$$

La razón es que para valorar una acción hoy necesitamos conocer cuánto valen los estados futuros.



En:

$$
t=3
$$

ya no existe futuro después de tomar la decisión actual.

Entonces Bellman se simplifica a:

$$
V_3(x)=
\min_{a\in A(x)}
\bar c(x,a)
$$

porque:

$$
V_4(x)=0
$$

Esto significa que en la última etapa solamente importa minimizar el costo inmediato.



#  Política óptima

La política óptima se obtiene tomando la acción que minimiza:

$$
\bar c(x,a)
+
\sum_{y\in X}Q(y|x,a)V_{t+1}(y)
$$

Formalmente:

$$
\pi_t^*(x)
=
\arg\min_{a\in A(x)}
\left\{
\bar c(x,a)
+
\sum_{y\in X}Q(y|x,a)V_{t+1}(y)
\right\}
$$



#  Interpretación esperada

Como la penalización por faltante es relativamente alta:

$$
p=5
$$

es probable que emerja una política tipo umbral.

Es decir:
- cuando el inventario es muy bajo conviene ordenar,
- cuando el inventario ya es suficientemente alto conviene no ordenar.

Esto es exactamente lo que se suele observar en modelos clásicos de inventario.


In [14]:
# datos base del ejercicio 4
estados_ej4 = [0, 1, 2, 3]
horizonte_ej4 = 4
capacidad_ej4 = 3
orden_maxima_ej4 = 2

prob_demanda_ej4 = {
    0: 0.3,
    1: 0.4,
    2: 0.3,
}

costo_fijo_ej4 = 2
costo_unitario_ej4 = 1
costo_mantener_ej4 = 1
penalizacion_faltante_ej4 = 5

acciones_ej4, transiciones_ej4, costos_esperados_ej4 = construir_mdp_inventario(
    estados=estados_ej4,
    prob_demanda=prob_demanda_ej4,
    orden_maxima=orden_maxima_ej4,
    capacidad=capacidad_ej4,
    costo_fijo=costo_fijo_ej4,
    costo_unitario=costo_unitario_ej4,
    costo_mantener=costo_mantener_ej4,
    penalizacion_faltante=penalizacion_faltante_ej4,
)

acciones_df_ej4 = pd.DataFrame({
    "estado_actual": list(acciones_ej4.keys()),
    "acciones_factibles": list(acciones_ej4.values()),
})

acciones_df_ej4


,estado_actual,acciones_factibles
0,0,"[0, 1, 2]"
1,1,"[0, 1, 2]"
2,2,"[0, 1]"
3,3,[0]


In [15]:
# kernel del ejercicio 4 construido desde la demanda
kernel_df_ej4 = tabla_transiciones(estados_ej4, transiciones_ej4)
kernel_df_ej4


,estado_actual,accion,q_0,q_1,q_2,q_3
0,0,0,1.0,0.0,0.0,0.0
1,0,1,0.7,0.3,0.0,0.0
2,0,2,0.3,0.4,0.3,0.0
3,1,0,0.7,0.3,0.0,0.0
4,1,1,0.3,0.4,0.3,0.0
5,1,2,0.0,0.3,0.4,0.3
6,2,0,0.3,0.4,0.3,0.0
7,2,1,0.0,0.3,0.4,0.3
8,3,0,0.0,0.3,0.4,0.3


In [16]:
# costo esperado inmediato para cada par estado-accion
costos_esperados_df_ej4 = tabla_valores_inmediatos(
    valores_inmediatos=costos_esperados_ej4,
    nombre_columna="costo_esperado",
)

costos_esperados_df_ej4


,estado_actual,accion,costo_esperado
0,0,0,5.0
1,0,1,4.8
2,0,2,5.0
3,1,0,1.8
4,1,1,4.0
5,1,2,6.0
6,2,0,1.0
7,2,1,5.0
8,3,0,2.0


In [17]:
# backward induction para minimizar costo total esperado
valores_opt_ej4, politica_opt_ej4 = resolver_backward_induction(
    estados=estados_ej4,
    acciones_por_estado=acciones_ej4,
    transiciones=transiciones_ej4,
    valores_inmediatos=costos_esperados_ej4,
    horizonte=horizonte_ej4,
    objetivo="min",
)

print("funcion de valor optima")
display(tabla_de_valores(valores_opt_ej4))

print("policy optima")
display(tabla_de_politica(politica_opt_ej4))


funcion de valor optima


,0,1,2,3
t,,,,
4,0.000,0.000,0.000,0.000
3,4.800,1.800,1.000,2.000
2,7.460,5.700,3.460,3.540
1,10.556,8.732,6.556,6.156
0,13.626,11.809,9.626,9.089


policy optima


,0,1,2,3
t,,,,
3,1,0,0,0
2,2,0,0,0
1,2,0,0,0
0,2,0,0,0


In [18]:
# detalle completo de bellman para ver que accion gano en cada caso
bellman_detalle_ej4 = detalle_bellman(
    estados=estados_ej4,
    acciones_por_estado=acciones_ej4,
    transiciones=transiciones_ej4,
    valores_inmediatos=costos_esperados_ej4,
    valores=valores_opt_ej4,
    horizonte=horizonte_ej4,
    nombre_inmediato="costo_inmediato",
    nombre_total="costo_total_accion",
)

bellman_detalle_ej4


,t,estado_actual,accion,costo_inmediato,valor_futuro_esperado,costo_total_accion
0,3,0,0,5.0,0.000,5.000
1,3,0,1,4.8,0.000,4.800
2,3,0,2,5.0,0.000,5.000
3,3,1,0,1.8,0.000,1.800
4,3,1,1,4.0,0.000,4.000
5,3,1,2,6.0,0.000,6.000
6,3,2,0,1.0,0.000,1.000
7,3,2,1,5.0,0.000,5.000
8,3,3,0,2.0,0.000,2.000
9,2,0,0,5.0,4.800,9.800


In [19]:
# ultima etapa: como v_4 = 0, aqui basicamente se compara costo inmediato
ultima_etapa_ej4 = bellman_detalle_ej4[bellman_detalle_ej4["t"] == horizonte_ej4 - 1]
ultima_etapa_ej4


,t,estado_actual,accion,costo_inmediato,valor_futuro_esperado,costo_total_accion
0,3,0,0,5.0,0.0,5.0
1,3,0,1,4.8,0.0,4.8
2,3,0,2,5.0,0.0,5.0
3,3,1,0,1.8,0.0,1.8
4,3,1,1,4.0,0.0,4.0
5,3,1,2,6.0,0.0,6.0
6,3,2,0,1.0,0.0,1.0
7,3,2,1,5.0,0.0,5.0
8,3,3,0,2.0,0.0,2.0


In [20]:
# lectura tipo umbral: miro desde que inventario la policy deja de pedir
policy_tabla_ej4 = tabla_de_politica(politica_opt_ej4)

analisis_umbral_ej4 = pd.DataFrame({
    "t": list(policy_tabla_ej4.index),
    "acciones_por_estado": [policy_tabla_ej4.loc[t].to_dict() for t in policy_tabla_ej4.index],
    "primer_estado_con_orden_cero": [
        min([estado for estado in estados_ej4 if politica_opt_ej4[t][estado] == 0])
        for t in policy_tabla_ej4.index
    ],
})

analisis_umbral_ej4


,t,acciones_por_estado,primer_estado_con_orden_cero
0,3,"{0: 1, 1: 0, 2: 0, 3: 0}",1
1,2,"{0: 2, 1: 0, 2: 0, 3: 0}",1
2,1,"{0: 2, 1: 0, 2: 0, 3: 0}",1
3,0,"{0: 2, 1: 0, 2: 0, 3: 0}",1


In [21]:
# comparo la policy del periodo inicial contra la del periodo final
comparacion_periodos_ej4 = pd.DataFrame({
    "estado": estados_ej4,
    "accion_t0": [politica_opt_ej4[0][estado] for estado in estados_ej4],
    "accion_t_final": [politica_opt_ej4[horizonte_ej4 - 1][estado] for estado in estados_ej4],
})

comparacion_periodos_ej4


,estado,accion_t0,accion_t_final
0,0,2,1
1,1,0,0
2,2,0,0
3,3,0,0


In [22]:
# monte carlo de la policy optima, usando demanda simulada de verdad
estado_inicial_ej4 = 0
n_sim_ej4 = 5000

media_mc_opt_ej4, costos_mc_opt_ej4, trayectorias_opt_ej4 = simular_politica_inventario(
    politica=politica_opt_ej4,
    estado_inicial=estado_inicial_ej4,
    horizonte=horizonte_ej4,
    prob_demanda=prob_demanda_ej4,
    costo_fijo=costo_fijo_ej4,
    costo_unitario=costo_unitario_ej4,
    costo_mantener=costo_mantener_ej4,
    penalizacion_faltante=penalizacion_faltante_ej4,
    n_sim=n_sim_ej4,
    seed=123,
    politica_depende_tiempo=True,
)

resumen_mc_ej4 = pd.DataFrame({
    "estado_inicial": [estado_inicial_ej4],
    "costo_analitico": [valores_opt_ej4[0][estado_inicial_ej4]],
    "costo_monte_carlo": [media_mc_opt_ej4],
})

resumen_mc_ej4.round(3)


,estado_inicial,costo_analitico,costo_monte_carlo
0,0,13.626,13.563


In [23]:
# trayectoria example de inventario con la policy optima
pd.DataFrame(trayectorias_opt_ej4[0])


,t,inventario_inicial,orden,demanda,inventario_final,costo
0,0,0,2,1,1,5
1,1,1,0,0,1,1
2,2,1,0,0,1,1
3,3,1,0,0,1,1


In [24]:
# valor optimo desde todos los estados iniciales
resumen_estados_ej4 = pd.DataFrame({
    "estado_inicial": estados_ej4,
    "costo_optimo_analitico": [valores_opt_ej4[0][estado] for estado in estados_ej4],
})

resumen_estados_ej4.round(3)


,estado_inicial,costo_optimo_analitico
0,0,13.626
1,1,11.809
2,2,9.626
3,3,9.089


In [25]:
# check final rapido para confirmar que todas las piezas principales existen
check_final_ej4 = pd.DataFrame({
    "pieza": ["kernel", "costos", "valores", "policy", "monte_carlo"],
    "ok": [
        not kernel_df_ej4.empty,
        not costos_esperados_df_ej4.empty,
        horizonte_ej4 in valores_opt_ej4,
        0 in politica_opt_ej4,
        len(costos_mc_opt_ej4) == n_sim_ej4,
    ],
})

check_final_ej4


,pieza,ok
0,kernel,True
1,costos,True
2,valores,True
3,policy,True
4,monte_carlo,True
